## Import

In [94]:
import numpy as np
import scipy
from scipy import signal
from scipy.signal import hilbert
from scipy import integrate
from pathlib import Path

try:
    from scipy.integrate import simps
except ImportError:
    from scipy.integrate import simpson

import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from matplotlib.colors import to_hex

import warnings
import re
import matplotlib.image as mpimg
import os
import shutil
import csv
import glob
import subprocess

from PIL import Image
from itertools import product

from scipy.integrate import solve_ivp
from scipy.interpolate import interp1d
from scipy.optimize import curve_fit

from functools import partial

# Полностью отключить все предупреждения
warnings.filterwarnings("ignore")

## get tree

In [85]:
result = subprocess.run(
    ["find", ".", "-mindepth", "6", "-maxdepth", "6", "-type", "d"],
    cwd="all_data/main",
    capture_output=True,
    text=True
)

# print(result.stdout)

all_data_folders = [
    line[2:] if line.startswith("./") else line
    for line in result.stdout.strip().split("\n")
]

# print(all_data_folders)

tree = {}

for path in all_data_folders:
    parts = path.split("/")
    
    levels = parts[:5]              # 5 уровней вложенности
    full_folder_name = path         # значение
    
    current = tree
    
    for part in levels[:-1]:        # идём до 4-го уровня
        current = current.setdefault(part, {})
    
    # 5-й уровень → присваиваем строку
    current[levels[-1]] = full_folder_name

# print(tree)

# tree['1eV']['sigma']['21fs']['4d10']['1d12']
# exists = tree.get('1eV', {}).get('sigma', {}).get('21fs', {}).get('4d10', {}).get('1d12') is not None

In [86]:
folder1 = 'all_data/main/'+tree['1eV']['sigma']['21fs']['4d10']['1d12']
folder2 = 'all_data/main/'+tree['1eV']['amorphous']['21fs']['4d10']['1d12']


folder3 = 'all_data/main/'+tree['1.5eV']['sigma']['21fs']['4d10']['5d11']
folder4 = []
if(tree.get('1.5eV', {}).get('amorphous', {}).get('21fs', {}).get('4d10', {}).get('5d11') is not None):
    folder4 = 'all_data/main/'+tree['1.5eV']['amorphous']['21fs']['4d10']['5d11']

folder5 = 'all_data/main/'+tree['2eV']['sigma']['21fs']['4d10']['6d11']
folder6 = 'all_data/main/'+tree['2eV']['amorphous']['21fs']['4d10']['6d11']

main_folders=[folder1, folder2, folder3, folder4, folder5, folder6]

## energy z

In [158]:
# for main_folder in main_folders:
#     if(main_folder):        

In [163]:
files_path = Path(main_folders[0]+'/current_energy/data')
filenames_arr=[]

for item in files_path.iterdir():
    filename = Path(item).name
    # print(filename)
    filenames_arr.append(filename)

filenames_arr = sorted(filenames_arr)
filenames_arr

['1_current_energy_data_c333_s0.025_r18_k16_a1_i804_t9_p4d10_1d12.csv',
 '2_current_energy_data_c333_s0.050_r18_k16_a1_i804_t9_p4d10_1d12.csv',
 '3_current_energy_data_c333_s0.075_r18_k16_a1_i804_t9_p4d10_1d12.csv',
 '4_current_energy_data_c333_s0.100_r18_k16_a1_i804_t9_p4d10_1d12.csv',
 '5_current_energy_data_c333_s0.125_r18_k16_a1_i804_t9_p4d10_1d12.csv',
 '6_current_energy_data_c333_s0.150_r18_k16_a1_i804_t9_p4d10_1d12.csv',
 '7_current_energy_data_c333_s0.175_r18_k16_a1_i804_t9_p4d10_1d12.csv',
 '8_current_energy_data_c333_s0.200_r18_k16_a1_i804_t9_p4d10_1d12.csv']

In [ ]:
j=0
for folder in sorted_folders:

    j=j+1
    
    # Загрузка данных, пропуская строки с комментариями (начинаются с #)
    data_both = pd.read_csv(
        folder + '/both_pulses_rt.data',
        comment='#',
        delim_whitespace=True,
        header=None
    )
    
    data_pump = pd.read_csv(
        folder + '/pump_pulse_rt.data',
        comment='#',
        delim_whitespace=True,
        header=None
    )
    
    data_probe = pd.read_csv(
        folder + '/probe_pulse_rt.data',
        comment='#',
        delim_whitespace=True,
        header=None
    )
    
    
    # Назначим читаемые имена колонкам
    data_both.columns = ['time_fs', 'Ac_ext_x', 'Ac_ext_y', 'Ac_ext_z', 'E_ext_x', 'E_ext_y', 'E_ext_z',         
        'Ac_tot_x', 'Ac_tot_y', 'Ac_tot_z', 'E_tot_x', 'E_tot_y', 'E_tot_z',         
        'Jm_x', 'Jm_y', 'Jm_z']
    
    data_pump.columns = ['time_fs', 'Ac_ext_x', 'Ac_ext_y', 'Ac_ext_z', 'E_ext_x', 'E_ext_y', 'E_ext_z',         
        'Ac_tot_x', 'Ac_tot_y', 'Ac_tot_z', 'E_tot_x', 'E_tot_y', 'E_tot_z',         
        'Jm_x', 'Jm_y', 'Jm_z']
    
    data_probe.columns = ['time_fs', 'Ac_ext_x', 'Ac_ext_y', 'Ac_ext_z', 'E_ext_x', 'E_ext_y', 'E_ext_z',         
        'Ac_tot_x', 'Ac_tot_y', 'Ac_tot_z', 'E_tot_x', 'E_tot_y', 'E_tot_z',         
        'Jm_x', 'Jm_y', 'Jm_z']
    
    
    t = data_both['time_fs']
    
    
    Jm_both_z  =  data_both['Jm_z']
    Jm_pump_z  =  data_pump['Jm_z']
    Jm_probe_z =  data_probe['Jm_z']
    
    Jm_both_x  =  data_both['Jm_x']
    Jm_pump_x  =  data_pump['Jm_x']
    Jm_probe_x =  data_probe['Jm_x']
    
    Jm_both_y  =  data_both['Jm_y']
    Jm_pump_y  =  data_pump['Jm_y']
    Jm_probe_y =  data_probe['Jm_y']
    
    
    El_f_pump_z = data_both['E_ext_z']
    El_f_probe_x = data_both['E_ext_x']
    
    
    
    product_both_z = El_f_pump_z * Jm_both_z * volume
    # Разность по времени
    dt = np.diff(t, prepend=t[0])  # prepend чтобы сохранить размер
    integral_both_z = np.cumsum(product_both_z * dt)
    
    product_pump_z = El_f_pump_z * Jm_pump_z * volume
    # Разность по времени
    dt = np.diff(t, prepend=t[0])  # prepend чтобы сохранить размер
    integral_pump_z = np.cumsum(product_pump_z * dt)
    
    product_probe_z = El_f_pump_z * Jm_probe_z * volume
    # Разность по времени
    dt = np.diff(t, prepend=t[0])  # prepend чтобы сохранить размер
    integral_probe_z = np.cumsum(product_probe_z * dt)
    
    product_delta_z = El_f_pump_z * (Jm_both_z - Jm_pump_z - Jm_probe_z) * volume
    # Разность по времени
    dt = np.diff(t, prepend=t[0])  # prepend чтобы сохранить размер
    integral_delta_z = np.cumsum(product_delta_z * dt)
    
    
    
    product_both_x = El_f_probe_x * Jm_both_x * volume
    # Разность по времени
    dt = np.diff(t, prepend=t[0])  # prepend чтобы сохранить размер
    integral_both_x = np.cumsum(product_both_x * dt)
    
    product_pump_x = El_f_probe_x * Jm_pump_x * volume
    # Разность по времени
    dt = np.diff(t, prepend=t[0])  # prepend чтобы сохранить размер
    integral_pump_x = np.cumsum(product_pump_x * dt)
    
    product_probe_x = El_f_probe_x * Jm_probe_x * volume
    # Разность по времени
    dt = np.diff(t, prepend=t[0])  # prepend чтобы сохранить размер
    integral_probe_x = np.cumsum(product_probe_x * dt)
    
    product_delta_x = El_f_probe_x * (Jm_both_x - Jm_pump_x - Jm_probe_x) * volume
    # Разность по времени
    dt = np.diff(t, prepend=t[0])  # prepend чтобы сохранить размер
    integral_delta_x = np.cumsum(product_delta_x * dt)
    
    
    
    
    
    fig, axs = plt.subplots(2, 2, figsize=(12, 5.5))

    for ax in axs.flat:
        ax.tick_params(axis='both', labelsize=11)
    
    line_w=1.4
    alpha_val=1
    color_field = color = mcolors.to_rgba("m", alpha=alpha_val)
    framealpha_val=0.95
    
    ax1 = axs[0][1]
    
    ax1.plot(t, -integral_both_x, color='b', label='both', linewidth=line_w)
    ax1.plot(t, -integral_pump_x, color='r', label='pump', linewidth=line_w)
    ax1.plot(t, -integral_probe_x, color='green', label='probe', linewidth=line_w)
    ax1.plot(t, -integral_delta_x, color='black', label='delta', linewidth=line_w)
    ax1.set_xlabel('Time [fs]', fontsize=12)
    ax1.set_ylabel('Energy [eV]', fontsize=12)
    ax1.set_title('Energy transfer along the x-axis', fontsize=14)
    ax1.legend(loc="upper right", framealpha=framealpha_val)
    ax1.grid()
    plt.tight_layout()
    
    
    ax1_right = ax1.twinx()
    ax1_right.plot(t, El_f_probe_x, color=color_field, label='probe field', linewidth=1.2)
    ax1_right.set_ylabel("Electric Field [V/Å]", color=color_field, fontsize=12)
    ax1_right.tick_params(axis="y", colors=color_field)
    ax1_right.spines["right"].set_color(color_field)
    # ax1_right.legend()
    plt.tight_layout()
    
    
    
    ax2 = axs[0][0]
    
    ax2.plot(t, -integral_both_z, color='b', label='both', linewidth=line_w)
    ax2.plot(t, -integral_pump_z, color='r', label='pump', linewidth=line_w)
    ax2.plot(t, -integral_probe_z, color='green', label='probe', linewidth=line_w)
    ax2.plot(t, -integral_delta_z, color='black', label='delta', linewidth=line_w)
    ax2.set_xlabel('Time [fs]', fontsize=12)
    ax2.set_ylabel('Energy [eV]', fontsize=12)
    ax2.set_title('Energy transfer along the z-axis', fontsize=14)
    ax2.legend(framealpha=framealpha_val)
    ax2.grid()
    plt.tight_layout()
    
    
    ax2_right = ax2.twinx()
    ax2_right.plot(t, El_f_pump_z, color=color_field, label='probe field', linewidth=1.2)
    ax2_right.set_ylabel("Electric Field [V/Å]", color=color_field, fontsize=12)
    ax2_right.tick_params(axis="y", colors=color_field)
    ax2_right.spines["right"].set_color(color_field)
    # ax1_right.legend()
    plt.tight_layout()
    
    
    
    ax3 = axs[1][0]
    
    ax3.plot(t, Jm_both_z, color='b', label='both', linewidth=line_w)
    ax3.plot(t, Jm_pump_z, color='r', label='pump', linewidth=line_w)
    ax3.plot(t, Jm_probe_z, color='green', label='probe', linewidth=line_w)
    # ax3.plot(t, -integral_delta_z, color='black', label='delta')
    ax3.set_xlabel('Time [fs]', fontsize=12)
    ax3.set_ylabel('J [$fs^{-1} \\cdot Å^{-2}$]', fontsize=12)
    ax3.set_title('Matter current density along the z-axis', fontsize=14)
    ax3.legend(framealpha=framealpha_val)
    ax3.grid()
    plt.tight_layout()
    
    
    # ax3_right = ax3.twinx()
    # ax3_right.plot(t, -El_f_pump_z, color='m', label='probe field')
    # ax3_right.set_ylabel("—Electric Field [V/Å]", color='m')
    # ax3_right.tick_params(axis="y", colors='m')
    # ax3_right.spines["right"].set_color('m')
    # # ax1_right.legend()
    # plt.tight_layout()
    
    
    
    ax4 = axs[1][1]
    
    ax4.plot(t, Jm_both_x, color='b', label='both', linewidth=line_w)
    ax4.plot(t, Jm_pump_x, color='r', label='pump', linewidth=line_w)
    ax4.plot(t, Jm_probe_x, color='green', label='probe', linewidth=line_w)
    ax4.plot(t, (Jm_both_x-Jm_pump_x-Jm_probe_x), color='black', label='delta', linewidth=line_w)
    ax4.set_xlabel('Time [fs]', fontsize=12)
    ax4.set_ylabel('J [$fs^{-1} \\cdot Å^{-2}$]', fontsize=12)
    ax4.set_title('Matter current density along the x-axis', fontsize=14)
    ax4.legend(framealpha=framealpha_val)
    ax4.grid()
    plt.tight_layout()
    
    # ax4_right = ax4.twinx()
    # ax4_right.plot(t, -El_f_pump_z, color='m', label='probe field')
    # ax4_right.set_ylabel("—Electric Field [V/Å]", color='m')
    # ax4_right.tick_params(axis="y", colors='m')
    # ax4_right.spines["right"].set_color('m')
    # # ax1_right.legend()
    # plt.tight_layout()
    
    
    
    plt.savefig("all_data/current_energy/ims/energy_current/png/" + str(j)+ '_energy_current_' + folder + ".png", dpi=200)
    plt.savefig("all_data/current_energy/ims/energy_current/pdf/" + str(j)+ '_energy_current_' + folder + ".pdf", bbox_inches="tight")

    plt.show()
    plt.close()

    print("For " + folder + " energy&current plots are completed")

print(" ")